# Model B: Full Context (Environment + Seller/Location)

In this notebook, we add Seller-specific features and location data to see how much they improve our prediction over Model A. The difference is the "Seller Impact".

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Data with Model A predictions
df = pd.read_csv('../../datasets/processed/olist_reviews_with_preds.csv')

# 2. Re-create Model A features
df['log_price'] = np.log1p(df['price'])
df['log_distance'] = np.log1p(df['distance_km'])
df['freight_ratio'] = df['freight_value'] / (df['price'] + 1)
df['late_delay'] = df['is_late'] * df['delivery_delay_days']
df['price_x_delay'] = df['log_price'] * df['delivery_delay_days']

base_features = [
    'actual_delivery_days', 'delivery_delay_days', 'is_late', 'log_distance', 
    'log_price', 'freight_ratio', 'late_delay', 'price_x_delay', 
    'product_weight_g', 'product_photos_qty', 'n_items'
]

# Add Seller/Location Features
full_features = base_features + [
    'seller_state', 'customer_state', 'seller_total_orders', 
    'product_description_lenght', 'product_name_lenght'
]

X = df[full_features].copy()

# Fix: Fill NaNs separately
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())
X = X.fillna('Unknown')

y = df['review_score']

# 3. Target Encoding for Seller ID
seller_means = df.groupby('seller_id')['review_score'].mean()
X['seller_mean_score'] = df['seller_id'].map(seller_means)

# 4. Encode Categoricals
le = LabelEncoder()
X['seller_state'] = le.fit_transform(X['seller_state'].astype(str))
X['customer_state'] = le.fit_transform(X['customer_state'].astype(str))

# 5. Re-run LDA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.select_dtypes(include=[np.number]))
lda = LDA(n_components=4)
X_lda = lda.fit_transform(X_scaled, y.astype(int))

for i in range(4):
    X[f'lda_{i}'] = X_lda[:, i]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Model B Feature count: {X.shape[1]}")

Model B Feature count: 21


## 1. Train Model B (Full Context)

In [3]:
xgb_model_b = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42,
    early_stopping_rounds=50
)

xgb_model_b.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

y_pred_b = xgb_model_b.predict(X_test)

print(f"\nModel B RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_b)):.4f}")
print(f"Model B R2 Score: {r2_score(y_test, y_pred_b):.4f}")

[0]	validation_0-rmse:1.27701
[100]	validation_0-rmse:1.11590
[157]	validation_0-rmse:1.11587

Model B RMSE: 1.1157
Model B R2 Score: 0.2476


## 2. Compare Results and Calculate Seller Premium

In [4]:
# Add Model B predictions to df
df['full_prediction_B'] = xgb_model_b.predict(X)

# CALCULATE SELLER PREMIUM
# Premium = How much better/worse the review is than what the environment predicted
df['seller_service_premium'] = df['full_prediction_B'] - df['expected_review_A']

print("Top Sellers by Service Premium (The ones who overcome bad logistics):")
print(df.groupby('seller_id')['seller_service_premium'].mean().sort_values(ascending=False).head(10))

df.to_csv('../../datasets/processed/final_seller_impact_results.csv', index=False)
print("\nFinal results saved to final_seller_impact_results.csv")

Top Sellers by Service Premium (The ones who overcome bad logistics):
seller_id
9b1f656ca3060f8af17a50d8a024b533    2.202445
8a207b6080964f3c1fc9d35bd6096ba4    1.796245
0b36063d5818f81ccb94b54adfaebbf5    1.608015
a9415cf907dfa12f9ed18693244f5329    1.478456
4e480be820e37de1444325ff358c9296    1.298728
f8e2c1b59cd4225a3e4fd0d013a76b93    1.172434
5f7300535881cef3b625fd683cf7d145    1.150387
d52cbce9845184537284a23c3bc3da0e    1.131544
a3fa18b3f688ec0fca3eb8bfcbd2d5b3    1.130419
782f78b632a43875dd5b53e3e49c9564    1.130322
Name: seller_service_premium, dtype: float64

Final results saved to final_seller_impact_results.csv
